# 01 — EDA & Preprocessing: CICIDS2017 Network Intrusion Detection

This notebook loads the raw CICIDS2017 flow-based CSVs (CICFlowMeter output),
explores class distributions and feature characteristics, cleans the data,
and writes a processed dataset to `data/processed/` for use in the modeling
notebook.

**Dataset:** [CICIDS2017](https://www.unb.ca/cic/datasets/ids-2017.html) — Canadian
Institute for Cybersecurity, 2017. ~2.8M labeled network flows captured over
5 days, covering benign traffic and attacks: Brute Force (FTP/SSH), DoS/DDoS,
Heartbleed, Web Attacks (SQLi/XSS/Brute Force), Infiltration, Botnet, and
Port Scan.

**Before running this notebook:** download the "MachineLearningCSV"
(GeneratedLabelledFlows) zip from the link above and extract the 8 CSV files
into `data/raw/`.


In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_cicids2017
from preprocessing import clean_dataframe, add_label_columns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)


## 1. Load raw data

Concatenates all CSV files in `data/raw/`.

In [ ]:
df_raw = load_cicids2017()
df_raw.head()


In [ ]:
print(f"Shape: {df_raw.shape}")
print(f"\nColumns ({df_raw.shape[1]}):")
print(list(df_raw.columns))


## 2. Class distribution (raw)

CICIDS2017 is heavily imbalanced — the vast majority of flows are `BENIGN`,
with attack classes ranging from tens of thousands of rows (DoS Hulk,
PortScan, DDoS) down to single digits (Heartbleed). This imbalance drives
several of our later modeling decisions (stratified splitting, macro-F1 as
the primary metric, grouping ultra-rare classes).


In [ ]:
label_counts = df_raw["Label"].value_counts()
print(label_counts)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(x=label_counts.values, y=label_counts.index, ax=ax, orient="h")
ax.set_xscale("log")
ax.set_xlabel("Count (log scale)")
ax.set_title("CICIDS2017 — Raw Label Distribution")
plt.tight_layout()
plt.savefig("../outputs/figures/01_raw_label_distribution.png", dpi=150)
plt.show()


## 3. Data quality checks

Known issues in CICIDS2017's CICFlowMeter output:
- `Flow Bytes/s` and `Flow Packets/s` can be `Infinity` (division by a
  zero-duration flow).
- A non-trivial number of exact duplicate rows exist across the daily files.
- A few columns (e.g. `Fwd Header Length.1`, `Bwd Avg Bulk Rate`) are
  constant / near-constant and add no signal.

Let's quantify these before cleaning.


In [ ]:
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns

n_inf = np.isinf(df_raw[numeric_cols]).sum().sum()
n_nan = df_raw[numeric_cols].isna().sum().sum()
n_dupes = df_raw.duplicated().sum()

print(f"Inf values across numeric columns: {n_inf}")
print(f"NaN values across numeric columns: {n_nan}")
print(f"Exact duplicate rows: {n_dupes} ({n_dupes / len(df_raw):.3%} of data)")


## 4. Clean the data

Using `preprocessing.clean_dataframe`: drops inf/NaN rows, duplicates, and zero-variance columns.

In [ ]:
df_clean = clean_dataframe(df_raw)
print(f"\nFinal shape: {df_clean.shape}")


## 5. Build label targets

We define two targets for the modeling notebook:
- **`label_binary`**: `BENIGN` vs `ATTACK` — the headline intrusion-detection
  task (is this traffic malicious at all?).
- **`label_multiclass`**: the specific attack category, with classes that
  have fewer than `MIN_CLASS_COUNT` samples grouped into `"Other"` so that
  stratified train/test splitting remains possible (you can't stratify a
  class with only 4 examples and put some in train and some in test).


In [ ]:
df_labeled = add_label_columns(df_clean)

print(df_labeled["label_binary"].value_counts())
print()
print(df_labeled["label_multiclass"].value_counts())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_labeled["label_binary"].value_counts().plot(kind="bar", ax=axes[0])
axes[0].set_title("Binary label distribution")
axes[0].tick_params(axis="x", rotation=0)

mc_counts = df_labeled["label_multiclass"].value_counts()
sns.barplot(x=mc_counts.values, y=mc_counts.index, ax=axes[1], orient="h")
axes[1].set_xscale("log")
axes[1].set_title("Multiclass label distribution (log scale)")

plt.tight_layout()
plt.savefig("../outputs/figures/02_label_distributions.png", dpi=150)
plt.show()


## 6. Feature correlation overview

A quick look at correlation among numeric features — heavily correlated
features are expected (e.g. `Total Length of Fwd Packets` vs
`Subflow Fwd Bytes`), and tree-based models handle this fine, but it's
useful context for interpreting feature importances later.


In [ ]:
numeric_cols = df_labeled.select_dtypes(include=[np.number]).columns
corr = df_labeled[numeric_cols].sample(n=min(50000, len(df_labeled)), random_state=42).corr()

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax,
            xticklabels=False, yticklabels=False)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("../outputs/figures/03_correlation_heatmap.png", dpi=150)
plt.show()

# Highly correlated pairs (|r| > 0.95)
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={0: "corr", "level_0": "feature_1", "level_1": "feature_2"})
)
high_corr_pairs = high_corr_pairs[high_corr_pairs["corr"].abs() > 0.95]
print(f"{len(high_corr_pairs)} feature pairs with |correlation| > 0.95")
high_corr_pairs.sort_values("corr", key=abs, ascending=False).head(10)


## 7. Distribution of a few key features by label

Sanity check: do attack flows actually look different from benign flows on
some intuitive features? (e.g. PortScan should have very short/zero-payload
flows; DoS Hulk should have high packet rates.)


In [ ]:
key_features = ["Flow Duration", "Total Fwd Packets", "Flow Packets/s",
                 "SYN Flag Count"]
key_features = [c for c in key_features if c in df_labeled.columns]

fig, axes = plt.subplots(1, len(key_features), figsize=(5 * len(key_features), 5))
for ax, feat in zip(axes, key_features):
    sample = df_labeled.sample(n=min(20000, len(df_labeled)), random_state=1)
    sns.boxplot(data=sample, x="label_binary", y=feat, ax=ax, showfliers=False)
    ax.set_title(feat.strip())

plt.tight_layout()
plt.savefig("../outputs/figures/04_feature_distributions.png", dpi=150)
plt.show()


## 8. Save processed data

Written to `data/processed/cicids2017_clean.parquet` for the modeling notebook.

In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)
df_labeled.to_parquet("../data/processed/cicids2017_clean.parquet", index=False)
print(f"Saved {df_labeled.shape[0]:,} rows x {df_labeled.shape[1]} columns")
